###### Load Dependencies

In [1]:
from importlib import reload
import scalers
from data import *
from classifications import *
from imputers import *
from scalers import StandardScaler, NormalScaler
import data as db
import matplotlib.pyplot as plt
reload( db )
import pandas as pd
import warnings

In [2]:
hdr = '\r\n' + '-' * 120 + '\r\n'
nwln = '\r\n'
warnings.filterwarnings( 'ignore' )

###### Load Data

In [2]:
# Load Data
schedx_filepath = r'C:\Users\terry\source\repos\mathy\stores\excel\Combined Schedules.xlsx'
nominal_columns = [ 'CombinedSchedulesId', 'MainAccount', 'TreasurySymbol', 'AccountName',
					'LineName', 'Line', 'Subfunction', 'Classification', 'BudgetEnforcementCategory' ]
out_years = [ f'OY-{i}' for i in range( 1, 10 ) ]
numeric_columns = [ 'PY', 'CY', 'BY' ]
data_columns = [ 'MainAccount', 'LineName', 'Line' ]
all_columns = nominal_columns + numeric_columns
index_columns = [ 'CombinedSchedulesId', ]
drop_columns = [ 'CombinedSchedulesId', 'LineName', 'Subfunction', 'Classification',
				 'BudgetEnforcementCategory' ]
max_columns = data_columns + numeric_columns

# Read Excel, set index and load columns# Reload with corrected dtypes and padding for codes
dtype_dict = \
{
	'MainAccount': str,
	'TreasurySymbol': str
}

df_excel = pd.read_excel( schedx_filepath, usecols=all_columns, sheet_name='Data'  )
df_excel.reset_index( )
df_excel.round( 2 )
# Fix formatting: pad with leading zeros and split TreasurySymbol
df_excel[ 'MainAccount' ] = df_excel[ 'MainAccount' ].str.replace( 'A-', '', regex=False )
df_excel[ 'MainAccount' ] = df_excel[ 'MainAccount' ].str.zfill( 4 )
df_excel[ 'TreasurySymbol' ] = df_excel[ 'TreasurySymbol' ].str.replace( 'A-', '', regex=False )
df_excel[ 'Line' ] = df_excel[ 'Line' ].str.replace( 'L-', '', regex=False )
df_excel[ 'Line' ] = df_excel[ 'Line' ].str.zfill( 4 )
df_excel[ 'AgencyCode' ] = df_excel[ 'TreasurySymbol' ].str[ :3 ].str.zfill( 3 )
df_excel[ 'MainAccountCode' ] = df_excel[ 'TreasurySymbol' ].str[ 3: ].str.zfill( 4 )

df_excel[ numeric_columns ] = df_excel[ numeric_columns ].round( 2 )
df_dataset = df_excel[ all_columns ].copy( )
df_nominal = df_excel[ nominal_columns ].copy( )
df_numeric = df_excel[ numeric_columns ].copy( )
df_schedx = df_excel[ max_columns ].copy( )
sns.set_style( 'darkgrid' )

In [3]:
ds = db.DataSource( df=df_numeric, target='BY' )
data = ds.data
targets = ds.targets.to_numpy( )
training_data = ds.training_data
training_values = ds.training_values
testing_data = ds.testing_data
testing_values = ds.testing_values

In [4]:
# Standardized data
scaler = StandardScaler( )
standard_training = scaler.train_transform( training_data )
standard_testing = scaler.train_transform( testing_data )

# Normalized data
normal = NormalScaler( )
normal_training = normal.train_transform( training_data )
normal_testing = normal.train_transform( testing_data )

In [5]:
model = LinearRegression( )
model.train( X=data, y=targets )
training_prediction = model.project( standard_training , training_values )
testing_prediction = model.project( standard_testing, testing_values )
training_score = model.score( standard_training , training_values )
testing_score = model.score( standard_testing, testing_values )
training_analysis = model.analyze( standard_training , training_values )
testing_analysis = model.analyze( standard_testing, testing_values )

In [7]:
training_analysis

{'MSE': np.float64(2.7332829642789736e+22),
 'RMSE': np.float64(2.7332829642789736e+22),
 'R2': -0.002398212209104189,
 'VAR': 1.211175604254322e-11,
 'MAE': np.float64(3000000.048960053)}

In [8]:
testing_analysis

{'MSE': np.float64(5.149700703910455e+18),
 'RMSE': np.float64(5.149700703910455e+18),
 'R2': -0.053005410136001796,
 'VAR': 9.043873427927451e-10,
 'MAE': np.float64(4499999.997965129)}

array([[0.00196977, 0.0021839 , 0.00220955, ..., 0.00253547, 0.00253512,
        0.0025354 ],
       [0.00231063, 0.00233243, 0.00233491, ..., 0.00236433, 0.0023643 ,
        0.00236433],
       [0.00233435, 0.00234211, 0.002343  , ..., 0.00235339, 0.00235338,
        0.00235339],
       ...,
       [0.00233769, 0.00234328, 0.00234399, ..., 0.00235371, 0.0023537 ,
        0.00235371],
       [0.00233848, 0.00234357, 0.00234424, ..., 0.00235366, 0.00235365,
        0.00235366],
       [0.0002071 , 0.00032634, 0.00040011, ..., 0.05997735, 0.06003489,
        0.06004328]], shape=(2400, 426))